# Pemeriksaan dan Validasi Dataset (Kelompok 3)
**Mata Kuliah:** Machine Learning (Kelas C), Bapak Adi Purnawan

**Anggota:** Deliana Br Manalu (2305551036) . Ravi Arnan Irianto (2305551076) . Ezza Putra Wibawa (2305551144) . Devin (2305551173)

---

Tahap ini adalah **pemeriksaan awal dataset**: memastikan data benar dan bersih
sebelum masuk ke preprocessing lanjutan. Belum ada pemodelan di sini.

**Sumber data:** Stroke Prediction Dataset (Kaggle, fedesoriano),
5.110 rekam medis pasien.


## 1. Muat Data

Dataset dimuat dari file lokal `data/` yang sudah di-upload ke GitHub.


In [ ]:
import pandas as pd
import numpy as np

# Bisa juga pakai file lokal: ../data/healthcare-stroke-data.csv
URL_DATA = "https://raw.githubusercontent.com/ravi-arnan/machine-learning/main/data/healthcare-stroke-data.csv"
df = pd.read_csv(URL_DATA)

print("Dataset berhasil dimuat.")
print("Jumlah baris :", df.shape[0])
print("Jumlah kolom :", df.shape[1])


## 2. Tampilan Lima Baris Pertama


In [ ]:
df.head()


## 3. Deskripsi Setiap Kolom

Agar kita paham arti tiap kolom sebelum mengolahnya.


In [ ]:
deskripsi = {
    "id": "Nomor identitas unik pasien (tidak dipakai prediksi)",
    "gender": "Jenis kelamin: Male, Female, Other",
    "age": "Usia pasien dalam tahun",
    "hypertension": "Riwayat hipertensi: 0 = tidak, 1 = ya (sudah angka biner)",
    "heart_disease": "Riwayat penyakit jantung: 0 = tidak, 1 = ya (angka biner)",
    "ever_married": "Pernah menikah: Yes / No",
    "work_type": "Tipe pekerjaan: Private, Self-employed, Govt_job, children, Never_worked",
    "Residence_type": "Tipe tempat tinggal: Urban / Rural",
    "avg_glucose_level": "Kadar glukosa rata-rata dalam darah",
    "bmi": "Indeks massa tubuh",
    "smoking_status": "Status merokok: formerly smoked, never smoked, smokes, Unknown",
    "stroke": "Label target: 0 = tidak stroke, 1 = stroke (angka biner)",
}

for kolom, arti in deskripsi.items():
    print(f"  {kolom:20s} -> {arti}")


## 4. Struktur dan Tipe Data

Cek jumlah kolom, tipe data, dan gambaran awal nilai kosong via info().


In [ ]:
df.info()


## 5. Cek Data Kosong (Pakai Perulangan)

Gunakan fungsi perulangan untuk mengecek kolom mana yang punya nilai kosong, jumlahnya, dan persentasenya.


In [ ]:
def cari_data_kosong(dataframe):
    """Loop tiap kolom, cetak kolom yang punya NaN."""
    print("=" * 50)
    print("PEMERIKSAAN DATA KOSONG")
    print("=" * 50)
    
    total_baris = len(dataframe)
    ada_kosong = False
    
    for kolom in dataframe.columns:
        jumlah = dataframe[kolom].isna().sum()
        if jumlah > 0:
            persen = (jumlah / total_baris) * 100
            print(f"  {kolom}: {jumlah} baris kosong ({persen:.2f}%)")
            ada_kosong = True
    
    if not ada_kosong:
        print("  Tidak ada data kosong.")
    print("=" * 50)

cari_data_kosong(df)


### 5a. Missing Value Tersembunyi

Data kosong tidak selalu NaN. Kadang berupa teks seperti "Unknown", "N/A",
yang artinya sebenarnya data tidak diketahui.


In [ ]:
def cari_nilai_tersembunyi(dataframe):
    """Loop kolom teks, cari nilai yang kemungkinan adalah missing value tersamar."""
    print("=" * 50)
    print("NILAI KOSONG TERSEMBUNYI")
    print("=" * 50)
    
    nilai_curiga = ["Unknown", "unknown", "N/A", "NA", "?", "-", ""]
    total_baris = len(dataframe)
    ditemukan = False
    
    for kolom in dataframe.columns:
        if not pd.api.types.is_string_dtype(dataframe[kolom]):
            continue
        for nilai in nilai_curiga:
            jumlah = (dataframe[kolom] == nilai).sum()
            if jumlah > 0:
                persen = (jumlah / total_baris) * 100
                print(f"  {kolom}: {jumlah} baris ({persen:.2f}%) berisi '{nilai}'")
                ditemukan = True
    
    if not ditemukan:
        print("  Tidak ada nilai tersembunyi yang mencurigakan.")
    print("=" * 50)

cari_nilai_tersembunyi(df)


## 6. Statistik Dasar

Cek rentang, mean, dan standar deviasi kolom numerik.


In [ ]:
df.describe().T.round(2)


## 7. Cek Duplikat (Pakai Perulangan)

Pastikan tidak ada baris yang duplikat.


In [ ]:
def cari_duplikat(dataframe):
    """Cek duplikat baris dan ID."""
    print("=" * 50)
    print("PEMERIKSAAN DUPLIKAT")
    print("=" * 50)

    # Duplikat baris penuh via duplicated()
    total = dataframe.duplicated().sum()
    print(f"  Baris duplikat (penuh): {total}")

    if "id" in dataframe.columns:
        duplikat_id = dataframe["id"].duplicated().sum()
        unik = dataframe["id"].nunique()
        print(f"  ID duplikat: {duplikat_id}")
        print(f"  ID unik: {unik} dari {len(dataframe)} baris")
    print("=" * 50)

cari_duplikat(df)


## 8. Distribusi Kolom Kategorikal (Pakai Perulangan)

Lihat sebaran nilai tiap kolom teks.


In [ ]:
for kolom in df.columns:
    if not pd.api.types.is_string_dtype(df[kolom]):
        continue
    print(f"--- {kolom} ---")
    print(df[kolom].value_counts().to_string())
    print()


## 9. Distribusi Kelas Target

Cek apakah data seimbang antara pasien stroke dan tidak.


In [ ]:
jumlah = df["stroke"].value_counts()
persen = (jumlah / len(df) * 100).round(2)

print("Distribusi kelas target (stroke):")
print(f"  0 (Tidak stroke): {jumlah[0]} ({persen[0]}%)")
print(f"  1 (Stroke):       {jumlah[1]} ({persen[1]}%)")
print(f"  Rasio: {jumlah[0]/jumlah[1]:.1f} : 1")


## 10. Perbaikan Data Kosong

Dari pengecekan di atas:
- `bmi`: 201 baris NaN (3,93%)
- `smoking_status`: 1.544 baris "Unknown" (30,22%)

BMI diisi median. Smoking status Unknown dibiarkan sebagai kategori sendiri (30% terlalu besar untuk dihapus). Biarkan dataset tetap utuh 5.110 baris.


In [ ]:
print("Sebelum perbaikan:")
print(f"  BMI kosong: {df['bmi'].isna().sum()}")
print(f"  smoking_status = Unknown: {(df['smoking_status'] == 'Unknown').sum()}")

df_bersih = df.copy()
df_bersih["bmi"] = df_bersih["bmi"].fillna(df_bersih["bmi"].median())

print()
print("Setelah perbaikan:")
print(f"  BMI kosong: {df_bersih['bmi'].isna().sum()}")
print(f"  Total data kosong tersisa: {df_bersih.isna().sum().sum()}")


## 11. Verifikasi Final: Dataset Bersih (Pakai Perulangan)

Konfirmasi semua kolom sudah bersih dari nilai kosong.


In [ ]:
def verifikasi_bersih(dataframe):
    """Loop tiap kolom, pastikan tidak ada NaN tersisa."""
    print("=" * 50)
    print("VERIFIKASI FINAL")
    print("=" * 50)

    total_kosong = 0
    for kolom in dataframe.columns:
        jk = dataframe[kolom].isna().sum()
        total_kosong += jk
        if jk > 0:
            print(f"  [!] {kolom}: masih {jk} kosong")

    if total_kosong == 0:
        print("  Semua kolom bersih. Tidak ada data kosong.")
    print(f"  Ukuran dataset: {dataframe.shape}")
    print("=" * 50)

verifikasi_bersih(df_bersih)


## 12. Pemeriksaan Outlier (Nilai Tidak Masuk Akal)

Nilai ekstrem belum tentu salah, tapi perlu dicatat sebelum diproses.


In [ ]:
print("Rentang nilai tiap kolom numerik:")
for kolom in df.select_dtypes(include="number").columns:
    print(f"  {kolom}: {df[kolom].min()} - {df[kolom].max()}")

print()
print("Kasus mencurigakan:")
if "bmi" in df.columns:
    print(f"  BMI > 60: {(df['bmi'] > 60).sum()} baris")
print(f"  Usia < 2 tahun: {(df['age'] < 2).sum()} baris")
if "gender" in df.columns:
    print(f"  gender = 'Other': {(df['gender'] == 'Other').sum()} baris")


In [ ]:
# Baris dengan BMI sangat tinggi untuk diperiksa
df[df["bmi"] > 60][["age", "bmi", "avg_glucose_level", "stroke"]]


## 13. Simpan Dataset Bersih

Hasil akhir disimpan ke CSV agar notebook berikutnya bisa langsung pakai.


In [ ]:
import pathlib

folder = pathlib.Path("../artifacts")
folder.mkdir(exist_ok=True)
jalan = folder / "stroke_pemeriksaan_awal.csv"
df_bersih.to_csv(jalan, index=False)

print(f"Dataset tersimpan di: {jalan}")
print(f"Ukuran: {df_bersih.shape}")


## Ringkasan

Dataset sudah diverifikasi:
- 5.110 baris x 12 kolom, tidak ada duplikat
- Missing value BMI (201 baris) sudah diisi median
- smoking_status "Unknown" (1.544 baris) dibiarkan sebagai kategori
- Tidak ada data kosong tersisa
- Outlier tercatat: BMI > 60, usia < 2 tahun, gender "Other"
- Kelas target tidak seimbang (stroke 4,87%)

Dataset siap untuk preprocessing lanjutan dan pemilihan fitur.
